In [ ]:
запросы на импорт
импортировать ОС
импортировать повторно
из openai импорт OpenAI
из тавили импортировать TavilyClient
из dotenv импорта load_dotenv

#Загрузить переменные среды
load_dotenv()

# Настройте ключ API
API_KEY = os.getenv("API_KEY")
BASE_URL = os.getenv("BASE_URL")
MODEL_ID = os.getenv("MODEL_ID")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

os.environ['TAVILY_API_KEY'] = TAVILY_API_KEY

# Слово системной подсказки
AGENT_SYSTEM_PROMPT = """Вы умный помощник в путешествии. Ваша задача — проанализировать запрос пользователя и шаг за шагом решить проблему, используя доступные инструменты.

# Доступные инструменты:
- `get_weather(city: str)`: Запросить погоду в указанном городе в реальном времени.
- `get_attraction(city: str, Weather: str)`: поиск рекомендуемых туристических достопримечательностей по городу и погоде.

# Требования к выходному формату:
Каждый ваш ответ должен строго следовать следующему формату и включать пару мыслей и действий:

Мысль: [Ваш мыслительный процесс и следующие шаги]
Действие: [конкретное действие, которое вы хотите выполнить]

Формат Действия должен быть одним из следующих:
1. Вызовите инструмент: имя_функции(arg_name="arg_value")
2. Завершите задачу: Завершить [окончательный ответ]

# ВАЖНОЕ ПРИМЕЧАНИЕ:
- Одновременно выводится только одна пара Мысль-Действие.
- Действие должно быть на одной строке, не переноситься- Когда собрано достаточно информации для ответа на вопрос пользователя, необходимо использовать формат Действие: Готово.

Пожалуйста, начните!
"""

In [9]:
def get_weather(city: str) -> str:
    """
    Запросите информацию о реальной погоде, вызвав API wttr.in.
    """
    # Конечная точка API, запрашиваем данные в формате JSON
    URL = f"https://wttr.in/{city}?format=j1"
    
    попробуйте:
        # Инициируем сетевой запрос
        ответ = запросы.получить (URL)
        # Проверьте, равен ли код состояния ответа 200 (успех)
        ответ.raise_for_status()
        # Анализируем возвращенные данные JSON
        данные = ответ.json()
        
        #Извлекаем текущие погодные условиятекущее_состояние = данные['текущее_состояние'][0]
        Weather_desc = текущее_состояние['weatherDesc'][0]['значение']
        temp_c = текущее_состояние['temp_C']
        
        #Форматируем в естественный язык и возвращаем
        return f"Текущая погода в {city}: {weather_desc}, температура {temp_c} градусов Цельсия"
        
    кроме запросов.исключения.RequestException как e:
        # Обработка сетевых ошибок
        return f «Ошибка: возникла проблема с сетью при запросе погоды — {e}»кроме (KeyError, IndexError) как e:
        # Обработка ошибок анализа данных
        return f «Ошибка: не удалось проанализировать данные о погоде, возможно, название города неверно — {e}»

def get_attraction (город: ул, погода: ул) -> ул:
    """
    Используйте API-интерфейс поиска Tavily для поиска и получения оптимизированных рекомендаций по достопримечательностям в зависимости от города и погоды.
    """
    api_key = os.environ.get("TAVILY_API_KEY")

    если не api_key:
        вернуть «Ошибка: TAVILY_API_KEY не настроен».

    # Инициализируем клиент Tavily
    tavily = TavilyClient(api_key=api_key)#Создаем точный запрос
    query = f"'{city}' Рекомендации и причины, по которым стоит посетить туристические достопримечательности в погоду '{weather}'"
    
    попробуйте:
        # Вызов API, include_answer=True вернет исчерпывающий ответ
        ответ = tavily.search(query=query, search_length="basic", include_answer=True)
        
        # Результаты, возвращаемые Тавили, очень чисты и могут быть использованы напрямую
        если ответ.get("ответ"):
            вернуть ответ["ответ"]
        
        # Если исчерпывающего ответа нет, отформатируйте исходный результатformatted_results = []
        для результата в response.get("results", []):
            formatted_results.append(f"- {result['title']}: {result['content']}")
        
        если не formatted_results:
             return «К сожалению, не найдено подходящих рекомендаций по туристическим достопримечательностям».

        return "По результатам поиска для вас была найдена следующая информация:\n" + "\n".join(formatted_results)

    кроме исключения как e:
        return f «Ошибка: проблема с выполнением поиска по Тавили — {e}»# Поместите все функции инструмента в словарь для облегчения последующих вызовов
доступные_инструменты = {
    "get_weather": get_weather,
    «get_attraction»: get_attraction,
}
print("Определение функции инструмента завершено!")

✅ Определение функции инструмента завершено!


In [10]:
класс OpenAICompatibleClient:
    """
    Клиент для вызова любого LLM-сервиса, совместимого с интерфейсом OpenAI.
    """
    def __init__(self, model: str, api_key: str, base_url: str):
        self.model = модель
        self.client = OpenAI(api_key=api_key, base_url=base_url)

    def генерировать (self, Prompt: str, system_prompt: str) -> str:
        """Вызовите LLM API для генерации ответа."""
        print("Вызов большой языковой модели...")
        попробуйте:
            сообщения = [{'роль': 'система', 'контент': system_prompt},
                {'роль': 'пользователь', 'контент': приглашение}
            ]
            ответ = self.client.chat.completions.create(
                модель = self.model,
                сообщения = сообщения,
                поток = Ложь
            )
            ответ = ответ.выбор[0].message.content
            print("Большая языковая модель ответила успешно.")обратный ответ
        кроме исключения как e:
            print(f"Произошла ошибка при вызове LLM API: {e}")
            return «Ошибка: произошла ошибка при вызове службы языковой модели».

класс TravelAssistant:
    """
    Интеллектуальный помощник в путешествии
    """
    защита __init__(сам):
        self.llm = OpenAICompatibleClient(
            модель=MODEL_ID,
            api_key=API_KEY,
            base_url=BASE_URL
        )
        self.prompt_history = []сброс защиты (сам):
        """Сбросить историю разговоров"""
        self.prompt_history = []
    
    защита add_user_message(self, message: str):
        """Добавить сообщение пользователя в историю"""
        self.prompt_history.append(f"Запрос пользователя: {сообщение}")
    
    Защиту add_assistant_message(self, message: str):
        """Добавить сообщение помощника в историю"""
        self.prompt_history.append(сообщение)
    
    Защиту add_observation (сам, наблюдение: ул):"""Добавить наблюдение в историю"""
        self.prompt_history.append(f"Наблюдение: {наблюдение}")
print("Определение класса интеллектуального помощника завершено!")

✅ Определение класса умного помощника завершено!


In [11]:
защита display_conversation (история):
    """Красиво отображаемая история разговоров"""
    печать("\n" + "="*60)
    print("📝История разговора")
    печать("="*60)
    
    для i сообщение в перечислении (история, 1):
        if message.startswith("Запрос пользователя:"):
            print(f"\n👤 Пользователь [{i}]: {message[5:]}")
        elif message.startswith("Мысль:"):
            print(f"\n🤔 Думая [{i}]: {message[8:].strip()}")elif message.startswith("Действие:"):
            print(f"🛠️ действие [{i}]: {message[7:].strip()}")
        elif message.startswith("Наблюдение:"):
            print(f"📊 наблюдайте [{i}]: {message[12:].strip()}")
        еще:
            print(f"💬 сообщение [{i}]: {message}")
    
    print("="*60 + "\n")

защита parse_action(action_str):
    """Разобрать строку действия"""
    если action_str.startswith("Готово"):match = re.match(r"\w+\[(.*)\]", action_str)
        если совпадение:
            вернуть «Готово», {»ответ»: match.group(1)}
        return "finish", {"ответ": "Задание выполнено"}
    
    имя_инструмента_match = re.search(r"(\w+)\(", action_str)
    если не имя_инструмента_соответствие:
        вернуть Нет, {}
    
    имя_инструмента = имя_инструмента_match.group(1)
    args_match = re.search(r"\((.*)\)", action_str)если args_match:
        args_str = args_match.group(1)
        kwargs = dict(re.findall(r'(\w+)="([^"]*)"', args_str))
    еще:
        кваргс = {}
    
    вернуть имя_инструмента, kwargs
print("Определение функции отображения завершено!")

✅ Определение функции дисплея завершено!


In [12]:
def run_assistant(user_input, max_iterations=5, display=True):
    """
    Основная функция, запускающая туристический помощник
    
    Аргументы:
        user_input: проблемы с пользовательским вводом
        max_iterations: максимальное количество итераций
        display: отображать ли историю разговоров
    
    Возврат:
        кортеж: (окончательный ответ, полная история разговора)
    """
    помощник = TravelAssistant()
    Assistant.add_user_message(user_input)
    
    если отобразить:
        print(f"👤 Пользовательский ввод: {user_input}")печать("="*50)
    
    для меня в диапазоне (max_iterations):
        если отобразить:
            print(f"\n🔄 цикл {i+1}/{max_iterations}")
        
        # Создайте полную подсказку и позвоните в LLM
        full_prompt = "\n".join(assistant.prompt_history)
        llm_output = Assistant.llm.generate(full_prompt, AGENT_SYSTEM_PROMPT)
        # Модель может выводить избыточные мысли-действия, и ее необходимо усечь.match = re.search(r'(Мысль:.*?Действие:.*?)(?=\n\s*(?:Мысль:|Действие:|Наблюдение:)|\Z)', llm_output, re.DOTALL)
        если совпадение:
            усечено = match.group(1).strip()
            если усечено!= llm_output.strip():
                llm_output = усечено
                print("⚠️ Лишние пары мысль-действие были усечены")
        
        Assistant.add_assistant_message(llm_output)если отобразить:
            print(f"🤖 вывод модели:\n{llm_output}")
        
        # действие анализа
        action_match = re.search(r"Действие: (.*)", llm_output, re.DOTALL)
        если не action_match:
            наблюдение = «Ошибка: невозможно проанализировать поле действия. Убедитесь, что ваш ответ строго соответствует формату «Мысль: ... Действие: ...».»
            замечание_str = f"Наблюдение: {наблюдение}"
            print(f"{observation_str}\n" + "="*40)Assistant.prompt_history.append(observation_str)
            продолжать
            
        action_str = action_match.group(1).strip()
        имя_инструмента, kwargs = parse_action(action_str)
        
        # Действие по завершению процесса
        если имя_инструмента == "Готово":
            Final_ответ = kwargs.get("ответ", "Задание выполнено")
            если отобразить:
                print(f"🎉 Миссия выполнена!")print(f"📋 Окончательный ответ: {final_answer}")
            вернуть окончательный_ответ, Assistant.prompt_history
        
        # Обработка вызовов инструментов
        если имя_инструмента в доступных_инструментах:
            если отобразить:
                print(f"🛠️ Вызов инструмента: {tool_name}({kwargs})")
            наблюдение = доступные_инструменты[имя_инструмента](**kwargs)
        еще:
            наблюдение = f"Ошибка: неопределенный инструмент '{tool_name}'"# Запись наблюдений
        если отобразить:
            print(f"📊 Результаты наблюдения: {observation}")
            печать("="*50)
        
        Assistant.add_observation(наблюдение)
    
    # Если достигнуто максимальное количество циклов и он все еще не завершен
    timeout_answer = "К сожалению, ваш запрос не был выполнен после нескольких попыток. Попробуйте упростить свой вопрос или повторите попытку позже."
    если отобразить:
        print(f"⏰ Достигнуто максимальное количество циклов: {timeout_answer}")
    
    вернуть timeout_ответ, Assistant.prompt_history

In [13]:
#Тестовый пример
защита test_basic_example():
    """Пример тестирования погоды в Пекине + рекомендации по достопримечательностям"""
    print("🚀 Начните тестировать пример погоды в Пекине + рекомендации по достопримечательностям")
    user_input = "Здравствуйте, пожалуйста, помогите мне узнать погоду в Пекине сегодня, а затем порекомендовать подходящую туристическую достопримечательность в зависимости от погоды."
    
    окончательный_ответ, история = run_assistant (user_input, display = True)
    
    печать("\n" + "="*60)
    print("📊 Тест завершен!")
    печать("="*60)
    print(f"Окончательный ответ: {final_answer}")
    
    # Показать полную историю разговоров
    display_conversation (история)вернуть окончательный_ответ, историю

# Запускаем тестовый пример
окончательный_ответ, история = test_basic_example()

🚀 Начните тестировать погоду в Пекине+Пример рекомендации по достопримечательностям
👤 пользовательский ввод: Здравствуйте, пожалуйста, помогите мне узнать погоду в Пекине сегодня, а затем порекомендовать подходящую туристическую достопримечательность в зависимости от погоды.。

🔄 цикл 1/5
Вызов большой языковой модели...
Большая языковая модель успешно ответила。
🤖 Выходные данные модели:
Thought: Сначала я проверю сегодняшнюю погоду в Пекине в режиме реального времени, затем подберу подходящие достопримечательности с учетом погоды и дам рекомендации.。
Action: get_weather(city="Пекин")
🛠️  Инструмент вызова: get_weather({'city': 'Пекин'})
📊 Наблюдения: Пекин текущая погода：Clear，температура-1градусы Цельсия

🔄 цикл 2/5
Вызов большой языковой модели...
Большая языковая модель успешно ответила。
🤖 Выходные данные модели:
Thought: Погода в Пекине сейчас солнечная, а температура-1°C，Подходит для осмотра достопримечательностей, но необходимо согреться; затем выберите подходящую рекомендацию из

In [14]:
защита интерактивного_travel_assistant():
    """
    Интерактивный помощник в путешествии
    """
    print("🌍 Добро пожаловать в умный помощник для путешествий!")
    print("💡 Вы можете запросить рекомендации по погоде и достопримечательностям в любом городе")
    print("❌ Чтобы завершить разговор, введите "quit" или "quit"\n")
    
    пока правда:
        user_input = input("👤 Пожалуйста, введите свой вопрос: ").strip()
        
        если user_input.lower() в ['quit', 'exit', 'exit']:
            print("👋 Спасибо за использование умного помощника по путешествиям, до свидания!")
            сломать
        
        если не user_input:print("⚠️ Пожалуйста, введите правильный вопрос")
            продолжать
        
        печать("\n" + "="*50)
        print("🔄 Обработка вашего запроса...")
        
        окончательный_ответ, история = run_assistant (user_input, display = True)
        
        print("\n🎯 Окончательный ответ:")
        печать("="*30)
        печать (final_ответ)
        печать("="*30)
        
        #Спросить, отображать ли полную историю разговоровshow_history = input("\n📖 Хотите показать полную историю разговоров? (да/нет): ").strip().lower()
        если show_history в ['y', 'yes', 'YES']:
            display_conversation (история)
        
        печать("\n" + "="*60)
        print("🔄 Готовьтесь к следующему вопросу...\n")

# Функция быстрого тестирования
def fast_test(city="Шанхай"):
    """Быстро проверить погоду и достопримечательности указанного города"""
    user_input = f"Пожалуйста, помогите мне узнать погоду в {city} и порекомендовать подходящие туристические достопримечательности"
    print(f"🚀 Быстрый тест: {user_input}")    final_answer, _ = run_assistant(user_input, display=True)
    return final_answer

In [17]:
# Основная запись запуска
если __name__ == "__main__":
    # Вы можете запустить тестовый пример напрямую
    print("Выберите режим работы:")
    print("1. Запуск тестового примера (Пекин)")
    print("2. Интерактивный режим")
    print("3. Быстро протестируйте другие города")
    
    choice = input("Пожалуйста, введите свой выбор (1/2/3): ").strip()
    
    если выбор == "1":
        test_basic_example()
    выбор элиф == "2":
        интерактивный_travel_assistant()
    выбор элиф == "3":
        city = input("Пожалуйста, введите город для тестирования: ").strip() или "Шанхай"быстрый_тест (город)
    еще:
        print("Неверный выбор, запустите тестовый пример...")
        test_basic_example()

Выберите режим работы:
1. Запустите тестовый пример (Пекин)
2. интерактивный режим
3. Быстро протестируйте другие города
🚀 быстрый тест: Пожалуйста, помогите мне узнать погоду в Гуанчжоу и порекомендуйте подходящие туристические достопримечательности.
👤 пользовательский ввод: Пожалуйста, помогите мне узнать погоду в Гуанчжоу и порекомендуйте подходящие туристические достопримечательности.

🔄 цикл 1/5
Вызов большой языковой модели...
Большая языковая модель успешно ответила。
🤖 Выходные данные модели:
Thought: Plan to fetch Guangzhouинформация о погоде в качестве первого шага, а затем фильтрация рекомендуемых достопримечательностей на основе погоды。  
Action: get_weather(city="Гуанчжоу")
🛠️  Инструмент вызова: get_weather({'city': 'Гуанчжоу'})
📊 Наблюдения: Текущая погода в Гуанчжоу：Clear，температура15градусы Цельсия

🔄 цикл 2/5
Вызов большой языковой модели...
Большая языковая модель успешно ответила。
🤖 Выходные данные модели:
Thought: GuangzhouТекущая погодаClear，Подходит для активного